<h1 align="center">Práctica 5B. Divide y vencerás (sesión 2): quicksort</h1>
<h3 style="display:block; margin-top:5px;" align="center">Estructuras de datos</h3>
<h3 style="display:block; margin-top:5px;" align="center">2026-2027</h3>
<h3 style="display:block; margin-top:5px;" align="center">Universitat Politècnica de València</h3>
<br>

**Pon/poned aquí tú/vuestros nombre(s):**
- (sustituir por el nombre)
- (sustituir por el nombre o borrar si se realiza individualmente)

## Introducción

En esta sesión vamos a estudiar experimentalmente cómo se comporta **quicksort** según cómo se elige el pivote y cómo se particiona el vector. Veremos que:

- con algunas elecciones del pivote y algunos tipos de vector, quicksort tiene **coste cuadrático**;
- en esos casos, además, puede **agotar la pila** de llamadas de Python;
- la partición de **Lomuto** vista en teoría funciona mal cuando hay muchos elementos repetidos, y la partición de **Hoare** lo resuelve;
- hacer la llamada recursiva **solo sobre la parte más pequeña** evita agotar la pila (pero no el coste cuadrático).

Tendrás que programar dos funciones:

1. **Actividad 1:** la partición de Hoare con mediana de tres.
2. **Actividad 2:** solo ejecutar experimentos ya preparados y observar los resultados.
3. **Actividad 3:** una versión de quicksort que hace la recursión sobre la parte más pequeña.

Para esta práctica necesitas el fichero `utils_practica5B.py` en la misma carpeta que este cuaderno. Contiene funciones auxiliares (tests y gráficas) que no hace falta leer.

## Código de partida

Es el código visto en teoría, con dos cambios:

- `quicksort_recurs` recibe como último parámetro **la función de partición** a utilizar. Así podemos probar quicksort con distintas particiones sin repetir código.
- `quicksort_recurs` **devuelve el número de comparaciones** realizadas, aproximadamente. Particionar un subvector de $m$ elementos cuesta unas $m$ comparaciones, así que cada llamada suma el tamaño del subvector que particiona más lo que devuelven las dos llamadas recursivas. Las comparaciones de `insertionSort` con 3 elementos o menos no se cuentan.

Además se añade `partition_random`, que elige un pivote aleatorio, lo lleva al final del subvector y usa la partición de Lomuto.

Ejecuta la celda:

In [ ]:
import random

def partition(A, first, last):
    # partición de Lomuto: el pivote es el último elemento
    i, pivot = first-1, A[last]
    for j in range(first, last+1):
        if A[j] <= pivot:
            i += 1
            A[i], A[j] = A[j], A[i]
    return i

def partition_random(A, first, last):
    # pivote aleatorio: lo llevamos al final y usamos Lomuto
    r = random.randint(first, last)
    A[r], A[last] = A[last], A[r]
    return partition(A, first, last)

def partition_median_of_3(A, first, last):
    # asume last - first >= 2
    middle = (first+last)//2
    A[first],A[middle],A[last] = sorted([A[first],A[middle],A[last]])
    # intercambiamos penúltimo con el pivote:
    A[middle],A[last-1] = A[last-1],A[middle]
    return partition(A,first+1,last-1)

def insertionSort(A, first, last):
    for index in range(first+1,last+1):
        key = A[index]
        pos = index
        while pos>first and A[pos-1]>key:
            A[pos]=A[pos-1]
            pos = pos-1
        A[pos]=key

def quicksort_recurs(A, first, last, particion):
    if last - first < 3:
        insertionSort(A, first, last)
        return 0
    p = particion(A, first, last)
    comparaciones = last - first + 1
    comparaciones += quicksort_recurs(A, first, p-1, particion)
    comparaciones += quicksort_recurs(A, p+1, last, particion)
    return comparaciones

La siguiente celda importa las funciones auxiliares y comprueba que el código de partida funciona:

In [ ]:
from utils_practica5B import probar_particion, probar_quicksort, grafica_coste

probar_particion(partition_median_of_3)
probar_quicksort(quicksort_recurs, partition_median_of_3)

v = [random.randrange(1000) for _ in range(500)]
print("Comparaciones para ordenar 500 elementos aleatorios:", quicksort_recurs(v, 0, len(v)-1, partition_median_of_3))

## Actividad 1: Partición de Hoare con mediana de tres

### El problema de Lomuto

Ejecuta la siguiente celda, que aplica la partición de Lomuto con mediana de tres a un vector de 12 elementos **iguales**:

In [ ]:
A = [7]*12
p = partition_median_of_3(A, 0, len(A)-1)
print("Lomuto + mediana de 3 con 12 elementos iguales -> el pivote queda en la posición", p)

El pivote queda en la posición 10, así que quicksort hará una llamada recursiva sobre 10 elementos y otra sobre 1. La mediana de tres no ayuda: los tres candidatos son iguales. El problema está en la partición: Lomuto usa `A[j] <= pivot`, de modo que **todos** los elementos iguales al pivote acaban a su izquierda.

**Pregunta:** si el vector tiene $n$ elementos iguales, ¿cuántos niveles de recursión hará quicksort aproximadamente?

*Respuesta:*

### La partición de Hoare

La partición de Hoare recorre el subvector con **dos índices**, `i` desde la izquierda y `j` desde la derecha, que avanzan uno hacia el otro. Cuando `i` encuentra un elemento que debería estar a la derecha y `j` uno que debería estar a la izquierda, los intercambia.

Vamos a combinarla con la mediana de tres. Tras ordenar `A[first]`, `A[middle]` y `A[last]` y llevar la mediana a `A[last-1]` (esa parte ya está hecha en el código) sabemos que:

- `A[first] <= pivot` y `A[last] >= pivot`, así que ya están en el lado correcto;
- el pivote está en `A[last-1]`;
- solo queda por particionar `A[first+1..last-2]`.

El algoritmo es:

1. `i` empieza en `first` y `j` en `last-1`.
2. Repetir:
    - avanzar `i` una posición y seguir avanzando **mientras** `A[i] < pivot`;
    - retroceder `j` una posición y seguir retrocediendo **mientras** `A[j] > pivot`;
    - si `i >= j`, los índices se han cruzado: salir del bucle;
    - si no, intercambiar `A[i]` y `A[j]`.
3. Colocar el pivote en su sitio intercambiando `A[i]` con `A[last-1]` y devolver `i`.

Ejemplo con `pivot = 5` (los elementos de los extremos ya los ha colocado la mediana de tres):

```
 índice:  0  1  2  3  4  5  6  7
 A:      [2, 8, 1, 5, 9, 3, 5, 9]      A[6]=5 es el pivote
             i           j             i se para en A[1]=8 y j en A[5]=3: intercambiar
 A:      [2, 3, 1, 5, 9, 8, 5, 9]
                   i                   i se para en A[3]=5 (igual al pivote)
                   j                   j se para en A[3]=5 (igual al pivote)
                                       i >= j: salir del bucle
 A:      [2, 3, 1, 5, 9, 8, 5, 9]      intercambiar A[3] con A[6] y devolver 3
```

Dos detalles importantes:

- **Los bucles se paran también en los elementos iguales al pivote**, porque las comparaciones son estrictas (`<` y `>`). Intercambiar dos elementos iguales parece inútil, pero es precisamente lo que reparte los iguales entre las dos mitades.
- **Los bucles internos no comprueban si `i` y `j` se salen del subvector.** No hace falta: `i` se parará como muy tarde en `A[last-1]`, que es el pivote, y `j` se parará como muy tarde en `A[first]`, que es `<= pivot`. Estos elementos que hacen de tope se llaman **centinelas**.

Completa la función:

In [ ]:
def partition_hoare_median_of_3(A, first, last):
    # asume last - first >= 2
    middle = (first+last)//2
    A[first],A[middle],A[last] = sorted([A[first],A[middle],A[last]])
    A[middle],A[last-1] = A[last-1],A[middle]
    pivot = A[last-1]
    i, j = first, last-1
    # COMPLETAR: bucle principal de la partición de Hoare
    # COMPLETAR: colocar el pivote en su sitio y devolver su posición
    raise NotImplementedError("Completa partition_hoare_median_of_3")

Comprueba que tu partición es correcta, que quicksort ordena bien con ella y dónde queda ahora el pivote con 12 elementos iguales:

In [ ]:
probar_particion(partition_hoare_median_of_3)
probar_quicksort(quicksort_recurs, partition_hoare_median_of_3)

A = [7]*12
p = partition_hoare_median_of_3(A, 0, len(A)-1)
print("Hoare + mediana de 3 con 12 elementos iguales -> el pivote queda en la posición", p)

## Actividad 2: Experimentos

En esta actividad **no tienes que programar nada**: ejecuta cada celda y fíjate en el resultado. Vamos a comparar cuatro variantes de partición:

In [ ]:
VARIANTES = {
    'Lomuto, pivote último':    partition,
    'Lomuto, pivote aleatorio': partition_random,
    'Lomuto, mediana de 3':     partition_median_of_3,
    'Hoare, mediana de 3':      partition_hoare_median_of_3,
}

### Experimento 1: desbordamiento de la pila

Cada llamada recursiva ocupa espacio en la **pila** de llamadas. Python limita la profundidad de recursión a unas 1000 llamadas; si se supera, lanza la excepción `RecursionError`.

La función `cabe_en_pila` intenta ordenar un vector y nos dice si se ha desbordado la pila. La celda siguiente la prueba con cada variante, con un vector ordenado y con un vector de elementos iguales de talla `n = 1500`:

In [ ]:
import sys
print("Límite de recursión de Python:", sys.getrecursionlimit())

def cabe_en_pila(quicksort, particion, A):
    try:
        quicksort(A, 0, len(A)-1, particion)
        return "ok"
    except RecursionError:
        return "DESBORDA"
    except NotImplementedError:
        return "sin hacer"

n = 1500
print(f"{'n = ' + str(n):26s} {'ordenado':>10s} {'iguales':>10s}")
for nombre, particion in VARIANTES.items():
    ordenado = cabe_en_pila(quicksort_recurs, particion, list(range(n)))
    iguales = cabe_en_pila(quicksort_recurs, particion, [7]*n)
    print(f"{nombre:26s} {ordenado:>10s} {iguales:>10s}")

Lomuto con mediana de tres no desborda con 1500 elementos iguales. Veamos qué ocurre con 3000:

In [ ]:
n = 3000
print(f"{'n = ' + str(n):26s} {'ordenado':>10s} {'iguales':>10s}")
for nombre, particion in VARIANTES.items():
    ordenado = cabe_en_pila(quicksort_recurs, particion, list(range(n)))
    iguales = cabe_en_pila(quicksort_recurs, particion, [7]*n)
    print(f"{nombre:26s} {ordenado:>10s} {iguales:>10s}")

**Pregunta:** ¿por qué Lomuto con mediana de tres necesita el doble de elementos iguales que Lomuto con pivote último para desbordar la pila?

*Respuesta:*

### Experimento 2: coste con vectores aleatorios

La función `grafica_coste` dibuja el número de comparaciones que devuelve quicksort con cada variante, para tallas de 100 a 800 (por debajo del límite de la pila). La línea discontinua es $n^2/2$, el coste del peor caso de quicksort, como referencia.

In [ ]:
grafica_coste(quicksort_recurs, VARIANTES, 'aleatorio')

Con vectores aleatorios todas las variantes hacen un número de comparaciones muy parecido y muy alejado de $n^2/2$: es el coste $O(n \log n)$ del caso medio.

### Experimento 3: coste con vectores ordenados

In [ ]:
grafica_coste(quicksort_recurs, VARIANTES, 'ordenado')

**Pregunta:** ¿qué variante tiene coste cuadrático con un vector ordenado? ¿Por qué?

*Respuesta:*

### Experimento 4: coste con vectores de elementos iguales

In [ ]:
grafica_coste(quicksort_recurs, VARIANTES, 'iguales')

Ojo: hay curvas exactamente superpuestas (si no ves una variante, está debajo de otra).

**Pregunta:** ¿qué variantes tienen coste cuadrático con todos los elementos iguales? ¿Cuál es la única de las cuatro variantes que funciona bien en los tres experimentos?

*Respuesta:*

## Actividad 3: Reducir la profundidad de recursión de quicksort

En la versión recursiva de quicksort realizamos dos llamadas recursivas después de cada partición: una para ordenar la parte izquierda y otra para ordenar la derecha.

Cuando las particiones están muy desequilibradas, la profundidad de recursión puede alcanzar $O(n)$, lo que puede provocar que se agote la pila de llamadas.

Podemos evitarlo modificando la forma en que procesamos las dos particiones:

1. **Parte más pequeña:** la ordenamos mediante una llamada recursiva, como en el quicksort original.
2. **Parte más grande:** en lugar de realizar otra llamada recursiva, continuamos ordenándola dentro de la misma función mediante un bucle `while`. Para ello, actualizamos `first` o `last` para que delimiten únicamente esa parte.

**¿Qué ventaja tiene?** Cada nueva llamada recursiva procesa, como máximo, la mitad de los elementos de la llamada anterior. Por tanto, la profundidad de recursión queda limitada a $O(\log n)$, independientemente de cómo se elijan los pivotes.

Esta optimización reduce el consumo de memoria, pero **no cambia la complejidad temporal**, que puede seguir siendo $O(n^2)$ en el peor caso.

### ¿Qué debes implementar?

Completa la función `quicksort_while` siguiendo las indicaciones del código proporcionado:

- Mientras el subvector tenga **4 o más elementos**, particiónalo, ordena recursivamente la parte más pequeña y continúa con la parte más grande mediante el bucle.
- Cuando queden **3 elementos o menos**, utiliza `insertionSort` para ordenarlos.
- Devuelve el número de comparaciones estimadas: suma `last - first + 1` por cada partición realizada y las comparaciones devueltas por las llamadas recursivas. No contabilices las comparaciones de `insertionSort`.

In [ ]:
def quicksort_while(A, first, last, particion):
    comparaciones = 0
    # COMPLETAR: mientras el subvector A[first..last] tenga 4 o más elementos:
    #   - particionar y sumar las comparaciones de la partición
    #   - llamar recursivamente sobre la parte MÁS PEQUEÑA y sumar lo que devuelve
    #   - actualizar first o last para seguir con la parte más grande
    # al salir del bucle quedan 3 elementos o menos: ordenarlos por inserción
    raise NotImplementedError("Completa quicksort_while")
    return comparaciones

Comprueba que ordena correctamente y que devuelve el mismo número de comparaciones que `quicksort_recurs` (con la misma partición y el mismo vector deben hacer exactamente las mismas particiones):

In [ ]:
probar_quicksort(quicksort_while, partition)
probar_quicksort(quicksort_while, partition_hoare_median_of_3)

v = [random.randrange(1000) for _ in range(500)]
print("quicksort_recurs:", quicksort_recurs(v[:], 0, len(v)-1, partition_median_of_3))
print("quicksort_while: ", quicksort_while(v[:], 0, len(v)-1, partition_median_of_3))

Repite ahora el experimento 1 con `quicksort_while` y `n = 3000`:

In [ ]:
n = 3000
print(f"{'n = ' + str(n):26s} {'ordenado':>10s} {'iguales':>10s}")
for nombre, particion in VARIANTES.items():
    ordenado = cabe_en_pila(quicksort_while, particion, list(range(n)))
    iguales = cabe_en_pila(quicksort_while, particion, [7]*n)
    print(f"{nombre:26s} {ordenado:>10s} {iguales:>10s}")

Ya no se desborda la pila. Pero ¿qué ha pasado con el coste? Ejecuta la celda:

In [ ]:
n = 3000
A = list(range(n))
print("Lomuto + pivote último, vector ordenado de", n, "elementos")
print("  comparaciones:", quicksort_while(A, 0, n-1, partition))
print("  n*n/2:        ", n*n//2)

Como se anunciaba al principio de la actividad, la pila ya no se desborda, pero el número de comparaciones sigue siendo $\approx n^2/2$: la recursión sobre la parte más pequeña resuelve el problema de la pila, no el del coste. El coste solo lo arregla una buena elección del pivote y de la partición (Hoare con mediana de tres).